In [83]:
from dataclasses import dataclass
import pickle
import gzip
import random
import numpy as np
from PIL import Image, ImageOps
from time import time

## Basic math functions
### Sigmoid function
The sigmoid function is defined as:
$$\sigma(x) = \frac{1}{1 + e^{-x}}$$
This function maps any real-valued number into the (0, 1) interval, making it useful for binary classification problems.

In [84]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [85]:
def sigmoid_prime(z):
    """Derivative of the sigmoid function."""
    return sigmoid(z) * (1 - sigmoid(z))

In [86]:
def cost_derivative(output_activations, y):
    """Return the vector of partial derivatives for the output activations."""
    return (output_activations - y)

In [87]:
@dataclass
class Network:
    num_layers: int
    biases: list
    weights: list

def init_network(layers):
    return Network(
        len(layers),
        [np.random.randn(y, 1) for y in layers[1:]],
        [np.random.randn(y, x) for x, y in zip(layers[:-1], layers[1:])]
    )

In [88]:
def feedforward(network, a):
    """Return the output of the network if ``a`` is input."""
    for b, w in zip(network.biases, network.weights):
        a = sigmoid(np.dot(w, a) + b)
    return a

In [89]:
def evaluate(network, test_data):
    """Return the number of test inputs for which the neural network outputs the correct result."""
    test_results = [(np.argmax(feedforward(network, x)), y) for (x, y) in test_data]
    return sum(int(x == y) for (x, y) in test_results)

In [90]:
def backprop(nn, x, y):
    nabla_b = [np.zeros(b.shape) for b in nn.biases]
    nabla_w = [np.zeros(w.shape) for w in nn.weights]

    # feedforward
    activation = x    # first layer activation is just its input
    activations = [x] # list to store all activations, layer by layer
    zs = []           # list to store all z vectors, layer by layer

    for b, w in zip(nn.biases, nn.weights):
        z = np.dot(w, activation) + b  # calculate z for the current layer
        zs.append(z)                   # store
        activation = sigmoid(z)        # layer output
        activations.append(activation) # store

    # backward pass

    # 1. starting from the output layer
    delta = cost_derivative(activations[-1], y) * sigmoid_prime(zs[-1]) 
    nabla_b[-1] = delta
    nabla_w[-1] = np.dot(delta, activations[-2].transpose())

    # 2. continue back to the input layer (i is the layer index, we're using i instead of l
    #    to improve readability -- l looks too much like 1)
    for i in range(2, nn.num_layers): # starting from the next-to-last layer
        z = zs[-i]
        sp = sigmoid_prime(z)
        delta = np.dot(nn.weights[-i + 1].transpose(), delta) * sp
        
        nabla_b[-i] = delta
        nabla_w[-i] = np.dot(delta, activations[-i - 1].transpose())
        
    return (nabla_b, nabla_w)

In [91]:
def update_mini_batch(network, mini_batch, eta):
    """Update the network's weights and biases by applying gradient descent using backpropagation to a single mini batch."""
    nabla_b = [np.zeros(b.shape) for b in network.biases]
    nabla_w = [np.zeros(w.shape) for w in network.weights]
    for x, y in mini_batch:
        delta_nabla_b, delta_nabla_w = backprop(network, x, y)
        nabla_b = [nb + dnb for nb, dnb in zip(nabla_b, delta_nabla_b)]
        nabla_w = [nw + dnw for nw, dnw in zip(nabla_w, delta_nabla_w)]
    network.weights = [w - (eta / len(mini_batch)) * nw
                       for w, nw in zip(network.weights, nabla_w)]
    network.biases = [b - (eta / len(mini_batch)) * nb
                      for b, nb in zip(network.biases, nabla_b)]

In [92]:
def stochastic_gradient_descent(network, training_data, epochs, mini_batch_size, eta, test_data=None):
    """Train the neural network using mini-batch stochastic gradient descent."""
    n = len(training_data)
    for j in range(epochs):
        random.shuffle(training_data)
        mini_batches = [training_data[k:k + mini_batch_size] for k in range(0, n, mini_batch_size)]
        for mini_batch in mini_batches:
            update_mini_batch(network, mini_batch, eta)
        if test_data:
            print(f"Epoch {j}: {evaluate(network, test_data)} / {len(test_data)}")
        else:
            print(f"Epoch {j} complete")

In [93]:
def learn(nn, training_data, epochs, mini_batch_size, learning_rate, test_data = None):
    n = len(training_data)

    for j in range(epochs):
        random.shuffle(training_data) # that's where "stochastic" comes from

        mini_batches = [
            training_data[k: k + mini_batch_size] for k in range(0, n, mini_batch_size)
        ]
        
        for mini_batch in mini_batches:
            update_mini_batch(nn, mini_batch, learning_rate) # that's where learning really happens

        if test_data:
            print('Epoch {0}: accuracy {1}%'.format(f'{j + 1:2}', 100.0 * evaluate(nn, test_data) / len(test_data)))
        else:
            print('Epoch {0} complete.'.format(f'{j + 1:2}'))

In [94]:
def load_data():
    f = gzip.open('mnist.pkl.gz', 'rb')
    training_data, validation_data, test_data = pickle.load(f, encoding="bytes")
    f.close()
    
    return (training_data, validation_data, test_data)

In [95]:
def one_hot_encode(y):
    """Return a one-hot encoded vector for the given digit."""
    e = np.zeros((10, 1))
    e[y] = 1.0
    return e

In [96]:
def load_data_wrapper():
    tr_d, va_d, te_d = load_data()
    
    training_inputs = [np.reshape(x, (784, 1)) for x in tr_d[0]]
    training_results = [one_hot_encode(y) for y in tr_d[1]]
    training_data = zip(training_inputs, training_results)
    validation_inputs = [np.reshape(x, (784, 1)) for x in va_d[0]]
    validation_data = zip(validation_inputs, va_d[1])
    test_inputs = [np.reshape(x, (784, 1)) for x in te_d[0]]
    test_data = zip(test_inputs, te_d[1])
    
    return (list(training_data), list(validation_data), list(test_data))

In [97]:
def print_shape(name, arr):
    try:
        # numpy arrays and lists of numpy arrays
        if hasattr(arr, "shape"):
            shape = arr.shape
        else:
            # handle nested lists (e.g., list of arrays)
            shape = type(arr).__name__
            try:
                shape = [a.shape for a in arr]
            except Exception:
                shape = str(shape)
        print(f"{name}: {shape}")
    except Exception as e:
        print(f"{name}: <could not determine shape> ({e})")

In [98]:
training_data, validation_data, test_data = load_data_wrapper() # load data

nn = init_network([784, 30, 10])

for l in range(0, nn.num_layers - 1):
    print('\nNetwork layer {0}'.format(l + 2)) # disregard the input layer
    print_shape('weights', nn.weights[l])
    print_shape('biases', nn.biases[l])
    
# hyper parameters
epochs = 15
mini_batch_size = 10
learning_rate = 3.0
    
print('\nLearning process started...\n')

time_start = time()

learn(nn, training_data, epochs, mini_batch_size, learning_rate, test_data)

time_end = time()

time_elapsed = time_end - time_start

print('\nLearning process complete in {0} seconds ({1} seconds per epoch)!\n'.format(f'{time_elapsed:.0f}', f'{time_elapsed / epochs:.1f}'))

print('Validation (with yet unseen data): accuracy {0}%'.format(100.0 * evaluate(nn, validation_data) / len(validation_data)))


Network layer 2
weights: (30, 784)
biases: (30, 1)

Network layer 3
weights: (10, 30)
biases: (10, 1)

Learning process started...

Epoch  1: accuracy 89.96%
Epoch  2: accuracy 91.34%
Epoch  3: accuracy 92.77%
Epoch  4: accuracy 93.06%
Epoch  5: accuracy 93.0%
Epoch  6: accuracy 93.19%
Epoch  7: accuracy 93.87%
Epoch  8: accuracy 94.06%
Epoch  9: accuracy 93.79%
Epoch 10: accuracy 94.09%
Epoch 11: accuracy 94.24%
Epoch 12: accuracy 94.41%
Epoch 13: accuracy 94.08%
Epoch 14: accuracy 94.4%
Epoch 15: accuracy 94.44%

Learning process complete in 116 seconds (7.7 seconds per epoch)!

Validation (with yet unseen data): accuracy 95.04%
